# Investigation: dFF artifact in binitmax run — ROI 103 (cell_roi_id 61, VISp_1)

**Session:** 755252_2024-11-19  
**Comparison:** `0002_cpos2_cneg4_lowess` vs `0010_cpos2_cneg4_lowess_binitmax`  
**Symptom:** F0trend traces look visually similar, but binitmax dFF shows values >±500 in the 200–500 s window.

The two recipes differ in only one field:
- `cpos2_cneg4`: `x0.b_init_from = "mean_F_minus_long_baseline"` → b_init ≈ mean(F − baseline_long) per ROI
- `cpos2_cneg4_binitmax`: `x0.b_init_from = "half_of_max_F_minus_min_F"` → b_init = (max(F) − min(F)) / 2 per ROI


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

ROI   = 102   # 0-indexed; ROI 103 in the GUI, cell_roi_id 61 in VISp_1
SESS  = '755252_2024-11-19'
INP   = f'/results/runs/first_try/{SESS}'
R1    = f'/results/runs/0002_cpos2_cneg4_lowess/{SESS}'
R2    = f'/results/runs/0010_cpos2_cneg4_lowess_binitmax/{SESS}'

ts    = np.load(f'{INP}/timestamps.npy')
F     = np.load(f'{INP}/F_all_array.npy', mmap_mode='r')[ROI].astype(np.float64)

# Trend stage output
t1    = np.load(f'{R1}/F0trend_all.npy', mmap_mode='r')[ROI].astype(np.float64)
t2    = np.load(f'{R2}/F0trend_all.npy', mmap_mode='r')[ROI].astype(np.float64)

# Full baseline (trend × LOWESS fluctuation)
f1    = np.load(f'{R1}/F0_all.npy', mmap_mode='r')[ROI].astype(np.float64)
f2    = np.load(f'{R2}/F0_all.npy', mmap_mode='r')[ROI].astype(np.float64)

# Fit parameter vectors
res1  = np.load(f'{R1}/res_all.npy')[ROI]
res2  = np.load(f'{R2}/res_all.npy')[ROI]
PARAMS = ['b_inf','b_slow','b_fast','b_bright','t_slow','t_fast','t_bright']

def safe_dff(F, b):
    safe = np.abs(b) > 1e-6
    return np.where(safe, (F - b) / np.where(safe, b, 1.0), 0.0)

dff_t1 = safe_dff(F, t1)
dff_t2 = safe_dff(F, t2)
dff_f1 = safe_dff(F, f1)
dff_f2 = safe_dff(F, f2)

# Fluctuation ratio  (F0 = F0trend × ratio when mode='ratio')
fluct1 = np.where(np.abs(t1) > 1e-6, f1 / t1, np.nan)
fluct2 = np.where(np.abs(t2) > 1e-6, f2 / t2, np.nan)

print('Loaded all arrays.')

## 1. Fitted parameter comparison

The optimizer converged to very different solutions depending on the b_init strategy.

In [ ]:
import pandas as pd

param_df = pd.DataFrame({'param': PARAMS, 'cpos2_cneg4': res1, 'binitmax': res2})
param_df['diff'] = param_df['binitmax'] - param_df['cpos2_cneg4']
param_df['ratio'] = param_df['binitmax'] / param_df['cpos2_cneg4'].replace(0, np.nan)
print(param_df.to_string(index=False, float_format='%.3g'))

## 2. What does b_init look like for each strategy?

For this ROI, binitmax gives a much larger b_init because the fluorescence trace has large transients.

In [ ]:
bl_long = np.load(f'{INP}/baseline_long_window_all_array.npy', mmap_mode='r')[ROI].astype(np.float64)

b_init_original  = float(np.mean(F - bl_long))
b_init_binitmax  = float((F.max() - F.min()) / 2.0)

print(f'b_init (mean_F_minus_long_baseline): {b_init_original:.3f}')
print(f'b_init (half_of_max_F_minus_min_F):  {b_init_binitmax:.3f}')
print(f'F range: [{F.min():.1f}, {F.max():.1f}]')
print(f'Ratio binitmax / original: {b_init_binitmax / b_init_original:.1f}x larger')

## 3. F0trend comparison — full trace

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# --- Raw F with both F0trends ---
ax = axes[0]
ax.plot(ts, F, color='k', lw=0.4, alpha=0.6, label='F')
ax.plot(ts, t1, color='steelblue', lw=1.5, label='F0trend (cpos2_cneg4)')
ax.plot(ts, t2, color='tomato',    lw=1.5, label='F0trend (binitmax)', ls='--')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_ylabel('Fluorescence (a.u.)')
ax.set_title('F and F0trend — ROI 103 (cell_roi_id 61, VISp_1)')
ax.legend(fontsize=8, loc='upper right')

# --- Difference F0trend_binitmax - F0trend_original ---
ax = axes[1]
ax.plot(ts, t2 - t1, color='purple', lw=1.0)
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_ylabel('Δ F0trend\n(binitmax − original)')
ax.set_title('F0trend difference (they are NOT identical despite looking similar)')

# --- F0trend binitmax zoom to show near-zero region ---
ax = axes[2]
ax.plot(ts, t1, color='steelblue', lw=1.5, label='F0trend (cpos2_cneg4)')
ax.plot(ts, t2, color='tomato',    lw=1.5, label='F0trend (binitmax)', ls='--')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_ylim(-2, 30)
ax.set_ylabel('F0trend (zoomed)')
ax.set_xlabel('Time (s)')
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Zoomed: binitmax F0trend dips near zero in 200–800 s')

for ax in axes:
    ax.axvspan(200, 500, alpha=0.08, color='red', label='_nolegend_')

plt.tight_layout()
plt.savefig('/root/capsule/code/03_fig1_F0trend_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. dFF from F0trend — why binitmax explodes in 200–500 s

dFF = (F − F0trend) / F0trend. When F0trend ≈ 0.3–1.6 (binitmax, near zero) and F has large transients (±100–300 a.u.), dFF becomes hundreds to thousands.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# --- F0trend in the 200-500 s window ---
ax = axes[0]
ax.plot(ts, t1, color='steelblue', lw=1.5, label='F0trend (cpos2_cneg4)')
ax.plot(ts, t2, color='tomato',    lw=1.5, label='F0trend (binitmax)', ls='--')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_ylabel('F0trend')
ax.set_title('F0trend values — binitmax dips to ~0.3 a.u. in the red zone')
ax.legend(fontsize=8)
ax.set_ylim(-1, 25)

# --- dFF from F0trend ---
ax = axes[1]
ax.plot(ts, np.clip(dff_t1, -20, 20), color='steelblue', lw=0.8, label='dFF·F0trend (cpos2_cneg4)', alpha=0.8)
ax.plot(ts, np.clip(dff_t2, -20, 20), color='tomato',    lw=0.8, label='dFF·F0trend (binitmax)', alpha=0.8, ls='--')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_ylabel('dFF (clipped ±20)')
ax.set_title('dFF from F0trend (clipped ±20 for display) — binitmax is ~5× larger in red zone')
ax.legend(fontsize=8)

# --- Actual unclipped dFF binitmax ---
ax = axes[2]
ax.plot(ts, dff_t2, color='tomato', lw=0.5, label='dFF·F0trend binitmax (unclipped)', alpha=0.9)
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_ylabel('dFF (unclipped)')
ax.set_xlabel('Time (s)')
ax.set_title('Unclipped binitmax dFF — extreme values up to ±600 in 200–500 s')
ax.legend(fontsize=8)

for ax in axes:
    ax.axvspan(200, 500, alpha=0.08, color='red')

plt.tight_layout()
plt.savefig('/root/capsule/code/03_fig2_dFF_F0trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Root cause: optimizer local minimum driven by b_init

The 7-param model is:
$$F_0^{\rm trend}(t) = b_{\rm inf} + b_{\rm slow} e^{-t/\tau_{\rm slow}} + b_{\rm fast} e^{-t/\tau_{\rm fast}} - b_{\rm bright} e^{-t/\tau_{\rm bright}}$$

When `b_init = half_of_max_F_minus_min_F` the optimizer starts with b_slow ≈ b_fast ≈ b_bright all equal to a **large value** (here ≈267 vs ≈14 for the original). The model has near-cancelling exponential terms `b_slow·E_slow − b_bright·E_bright`. With large amplitudes and nearly equal time constants (t_slow ≈ t_bright), the difference term passes through zero, creating a near-zero baseline in the middle of the trace.

In [ ]:
import sys
sys.path.insert(0, '/root/capsule/code/dff_baseline_search_qc_app')
from baseline_search.registry import biexp_bright_v1

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

model_pred1 = biexp_bright_v1(res1, ts)
model_pred2 = biexp_bright_v1(res2, ts)

# Decompose each model into its signed components
b_inf1, b_slow1, b_fast1, b_bright1, t_slow1, t_fast1, t_bright1 = res1
b_inf2, b_slow2, b_fast2, b_bright2, t_slow2, t_fast2, t_bright2 = res2

for col, (params, label, color) in enumerate([
    (res1, 'cpos2_cneg4 (original)', 'steelblue'),
    (res2, 'cpos2_cneg4_binitmax',   'tomato'),
]):
    b_inf, b_slow, b_fast, b_bright, t_slow, t_fast, t_bright = params
    E_slow   =  b_slow  * np.exp(-ts / t_slow)
    E_fast   =  b_fast  * np.exp(-ts / t_fast)
    E_bright = -b_bright * np.exp(-ts / t_bright)
    trend    = b_inf + E_slow + E_fast + E_bright

    ax = axes[0, col]
    ax.plot(ts, F, color='k', lw=0.4, alpha=0.4, label='F')
    ax.plot(ts, trend, color=color, lw=2, label='F0trend (model)')
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_title(f'{label}\nb_inf={b_inf:.1f}, b_slow={b_slow:.1f}, b_bright={b_bright:.1f}', fontsize=9)
    ax.set_ylabel('Fluorescence'); ax.legend(fontsize=7); ax.set_ylim(-300, 400)
    ax.axvspan(200, 500, alpha=0.08, color='red')

    ax = axes[1, col]
    ax.plot(ts, np.full_like(ts, b_inf), '--', color='gray', lw=1, label=f'b_inf={b_inf:.1f}')
    ax.plot(ts, E_slow,   color='green',  lw=1.5, label=f'+b_slow·exp(-t/τ_slow)   b={b_slow:.1f}, τ={t_slow:.0f}s')
    ax.plot(ts, E_fast,   color='orange', lw=1.5, label=f'+b_fast·exp(-t/τ_fast)   b={b_fast:.1f}, τ={t_fast:.0f}s')
    ax.plot(ts, E_bright, color='purple', lw=1.5, label=f'−b_bright·exp(-t/τ_bright)  b={b_bright:.1f}, τ={t_bright:.0f}s')
    ax.plot(ts, trend, color=color, lw=2, label='Sum (F0trend)', zorder=5)
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Component value')
    ax.legend(fontsize=6, loc='upper right'); ax.set_ylim(-200, 200)
    ax.axvspan(200, 500, alpha=0.08, color='red')

plt.suptitle('Model component decomposition — why binitmax dips near zero', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('/root/capsule/code/03_fig3_model_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Scatter: b_init strategies across all ROIs — how often does binitmax give extreme b_init?

In [ ]:
F_all   = np.load(f'{INP}/F_all_array.npy', mmap_mode='r').astype(np.float64)  # (N, T)
bl_long = np.load(f'{INP}/baseline_long_window_all_array.npy', mmap_mode='r').astype(np.float64)

b_init_orig = np.mean(F_all - bl_long, axis=1)
b_init_bmax = (F_all.max(axis=1) - F_all.min(axis=1)) / 2.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(b_init_orig, b_init_bmax, s=10, alpha=0.5, color='purple')
ax.scatter(b_init_orig[ROI], b_init_bmax[ROI], s=120, color='red', zorder=5, label=f'ROI {ROI+1}')
lim = max(b_init_bmax.max(), np.abs(b_init_orig).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
ax.set_xlabel('b_init: mean(F − baseline_long)')
ax.set_ylabel('b_init: (max(F) − min(F)) / 2')
ax.set_title('b_init strategy comparison across all ROIs')
ax.legend()

ax = axes[1]
ratio = b_init_bmax / np.where(b_init_orig > 0, b_init_orig, np.nan)
ax.hist(ratio[np.isfinite(ratio)], bins=50, color='slategray', edgecolor='white')
ax.axvline(ratio[ROI], color='red', lw=2, label=f'ROI {ROI+1}: ratio={ratio[ROI]:.1f}x')
ax.set_xlabel('binitmax / original b_init ratio')
ax.set_ylabel('Count')
ax.set_title('How much larger is binitmax b_init?')
ax.legend()

print(f'ROI {ROI+1}: b_init_orig={b_init_orig[ROI]:.2f}, b_init_bmax={b_init_bmax[ROI]:.2f}, ratio={ratio[ROI]:.1f}x')
print(f'Across all ROIs: median ratio={np.nanmedian(ratio):.2f}, 95th pct={np.nanpercentile(ratio[np.isfinite(ratio)], 95):.2f}')

plt.tight_layout()
plt.savefig('/root/capsule/code/03_fig4_binit_comparison_all_rois.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. F0trend minimum across all ROIs — how many dip near zero?

In [ ]:
t1_all = np.load(f'{R1}/F0trend_all.npy', mmap_mode='r').astype(np.float64)  # (N, T)
t2_all = np.load(f'{R2}/F0trend_all.npy', mmap_mode='r').astype(np.float64)

t1_min = t1_all.min(axis=1)
t2_min = t2_all.min(axis=1)

THRESHOLD = 5.0  # a.u. — F0trend below this is suspicious
n_bad1 = (t1_min < THRESHOLD).sum()
n_bad2 = (t2_min < THRESHOLD).sum()
N = t1_all.shape[0]

print(f'ROIs where min(F0trend) < {THRESHOLD} a.u.:')
print(f'  cpos2_cneg4 (original): {n_bad1}/{N} ({100*n_bad1/N:.1f}%)')
print(f'  binitmax:               {n_bad2}/{N} ({100*n_bad2/N:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(t1_min, t2_min, s=8, alpha=0.5, color='purple')
ax.scatter(t1_min[ROI], t2_min[ROI], s=120, color='red', zorder=5, label=f'ROI {ROI+1}')
mn, mx = min(t1_min.min(), t2_min.min()), max(t1_min.max(), t2_min.max())
ax.plot([mn, mx], [mn, mx], 'k--', lw=0.8)
ax.axhline(THRESHOLD, color='orange', lw=1, ls='--', label=f'threshold = {THRESHOLD}')
ax.axvline(THRESHOLD, color='orange', lw=1, ls='--')
ax.set_xlabel('min(F0trend) — original')
ax.set_ylabel('min(F0trend) — binitmax')
ax.set_title('Minimum F0trend value per ROI')
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(t1_min, bins=50, alpha=0.6, color='steelblue', label='original')
ax.hist(t2_min, bins=50, alpha=0.6, color='tomato',    label='binitmax')
ax.axvline(THRESHOLD, color='orange', lw=2, ls='--', label=f'threshold={THRESHOLD}')
ax.axvline(t2_min[ROI], color='red', lw=2, label=f'ROI {ROI+1} binitmax min={t2_min[ROI]:.2f}')
ax.set_xlabel('min(F0trend) (a.u.)')
ax.set_ylabel('Count')
ax.set_title('Distribution of min(F0trend) — binitmax has more near-zero cases')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/root/capsule/code/03_fig5_F0trend_min_all_rois.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

### Why does dFF blow up for binitmax in 200–500 s?

| Factor | cpos2_cneg4 (original) | cpos2_cneg4_binitmax |
|---|---|---|
| b_init | ≈ 14 a.u. | ≈ 267 a.u. (19×) |
| b_slow (fitted) | 20.3 | 131.7 |
| b_bright (fitted) | 57.4 | 170.1 |
| min(F0trend) | 2.05 | **0.28** |
| F0trend in 200–500 s | 2.1 – 3.0 | **0.28 – 1.6** |
| max |dFF from F0trend| in 200–500 s | ~82 | **~627** |

### Root cause

`b_init = (max(F) − min(F)) / 2` gives a **19× larger initial guess** for this ROI because the trace contains large transient excursions (F range ≈ ±267 a.u.). The optimizer uses this large b_init for `b_slow`, `b_fast`, and `b_bright` simultaneously.

The model is: $F_0^{\rm trend} = b_{\rm inf} + b_{\rm slow}e^{-t/\tau_{\rm slow}} - b_{\rm bright}e^{-t/\tau_{\rm bright}} + ...$

With b_slow ≈ 132 and b_bright ≈ 170 and nearly equal time constants (τ_slow ≈ 2155 s, τ_bright ≈ 1535 s), the `+b_slow·E_slow − b_bright·E_bright` pair partially cancels. The **difference** of two large exponentials with similar time constants passes through near-zero in the middle of the recording window (200–800 s).

This is an **optimizer local minimum**: the binitmax initialization pushes the fit into a different basin of attraction where the model overfits to the initial spike/transient structure, leaving a near-zero baseline in between.

### Implications for the sweep

- ROIs with high-amplitude transients (high F range) are particularly vulnerable — `binitmax` inflates b_init proportionally to those transients.
- This pathology is **not visible in dFF from F0** (after LOWESS) at this severity because the LOWESS step partially repairs it, but the near-zero F0trend propagates into a wrong fluctuation ratio.
- A simple QC flag: ROIs where `min(F0trend) < some threshold` (e.g. 5 a.u.) likely represent bad fits.
- The `mean_F_minus_long_baseline` b_init strategy is more robust for this ROI because it anchors the initial guess to the actual measured baseline rather than the full dynamic range.